OMNI 小时磁场 + 目标自回归 双时间尺度 ElasticNet 回归方案

In [10]:
# 导入库
import json
import warnings
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from pydantic import BaseModel, Field, ValidationError
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.switch_backend("Agg")

In [11]:
# 配置结构

class FileConfig(BaseModel):
    omni_file_path: Path
    lhaaso_file_path: Path | None = None
    omni_year_column: str
    omni_dayofyear_column: str
    omni_hour_column: str
    lhaaso_time_column: str | None = None
    target_column: str
    omni_feature_columns: List[str]
    lhaaso_feature_columns: List[str] = []

    def model_post_init(self, __context):
        if not self.omni_file_path.exists():
            raise FileNotFoundError(f"OMNI 文件不存在: {self.omni_file_path}")
        if self.lhaaso_feature_columns:
            if self.lhaaso_file_path is None:
                raise ValueError("启用 LHAASO 特征时必须提供 lhaaso_file_path")
            if self.lhaaso_time_column is None:
                raise ValueError("启用 LHAASO 特征时必须提供 lhaaso_time_column")
            if not self.lhaaso_file_path.exists():
                raise FileNotFoundError(f"LHAASO 文件不存在: {self.lhaaso_file_path}")

class HourLagConfig(BaseModel):
    var_name: str
    lag_list: List[int]

    def model_post_init(self, __context):
        if not self.lag_list:
            raise ValueError(f"小时变量 {self.var_name} 的 lag_list 不能为空")
        if any(lag < 1 for lag in self.lag_list):
            raise ValueError(f"小时变量 {self.var_name} 的 lag_list 必须全部大于等于 1")

class DayLagConfig(BaseModel):
    var_name: str
    lag_list: List[int]

    def model_post_init(self, __context):
        if not self.lag_list:
            raise ValueError(f"日尺度变量 {self.var_name} 的 lag_list 不能为空")
        if any(lag < 0 for lag in self.lag_list):
            raise ValueError(f"日尺度变量 {self.var_name} 的 lag_list 必须全部大于等于 0")

class BiWindowConfig(BaseModel):
    test_split_ratio: float = Field(ge=0.1, le=0.5)
    omni_hour_lags: List[HourLagConfig]
    lhaaso_day_lags: List[DayLagConfig] = []
    target_autoregressive_lags: List[int] = []

    def model_post_init(self, __context):
        if not self.omni_hour_lags:
            raise ValueError("biwindow_config.omni_hour_lags 不能为空")
        if self.lhaaso_day_lags is None:
            self.lhaaso_day_lags = []
        if any(lag < 1 for lag in self.target_autoregressive_lags):
            raise ValueError("target_autoregressive_lags 必须全部大于等于 1")

class ElasticNetConfig(BaseModel):
    alpha: float
    l1_ratio: float
    random_state: int

class ExperimentConfig(BaseModel):
    file_config: FileConfig
    biwindow_config: BiWindowConfig
    elasticnet_config: ElasticNetConfig

def load_llm_json_config(json_path: str | Path) -> ExperimentConfig:
    try:
        json_path = Path(json_path)
        raw_config = json_path.read_text(encoding="utf-8")
        config_dict = json.loads(raw_config)
        return ExperimentConfig.model_validate(config_dict)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"配置文件不存在: {json_path}") from e
    except json.JSONDecodeError as e:
        raise ValueError(f"配置文件不是合法 JSON: {e}") from e
    except ValidationError as e:
        raise ValueError(f"配置校验失败: {e}") from e

In [12]:
# 数据读取与双时间尺度对齐

def _read_table(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    if suffix in {".csv", ".txt"}:
        return pd.read_csv(file_path, sep=r"\s+|,", engine="python")
    raise ValueError(f"暂不支持的文件格式: {file_path}")

def _normalize_daily_time(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        normalized = pd.to_datetime(series, errors="coerce")
    else:
        cleaned = series.astype(str).str.replace(r"_x000d_", "", regex=True).str.strip()
        yyyymmdd = cleaned.str.extract(r"(\d{8})", expand=False)
        normalized = pd.to_datetime(yyyymmdd, format="%Y%m%d", errors="coerce")
        if normalized.isna().all():
            normalized = pd.to_datetime(cleaned, errors="coerce")

    if normalized.isna().any():
        bad_values = series[normalized.isna()].head(5).tolist()
        raise ValueError(f"LHAASO 时间列存在无法解析的日期值: {bad_values}")
    return normalized.dt.floor("D")

def _build_omni_hourly_time(df_omni: pd.DataFrame, file_cfg: FileConfig) -> pd.Series:
    for col in [file_cfg.omni_year_column, file_cfg.omni_dayofyear_column, file_cfg.omni_hour_column]:
        if col not in df_omni.columns:
            raise KeyError(f"OMNI 数据缺少时间列: {col}")

    year = df_omni[file_cfg.omni_year_column].astype(int)
    doy = df_omni[file_cfg.omni_dayofyear_column].astype(int)
    hour = df_omni[file_cfg.omni_hour_column].astype(int)
    base_time = pd.to_datetime(year.astype(str), format="%Y", errors="raise")
    hourly_time = base_time + pd.to_timedelta(doy - 1, unit="D") + pd.to_timedelta(hour, unit="h")
    return pd.Series(hourly_time, index=df_omni.index, name="omni_time")

def load_and_merge_biwindow_data(cfg: ExperimentConfig) -> pd.DataFrame:
    file_cfg = cfg.file_config
    df_omni = _read_table(file_cfg.omni_file_path)


    omni_required = list(dict.fromkeys(file_cfg.omni_feature_columns + [file_cfg.target_column]))
    lhaaso_required = list(dict.fromkeys(file_cfg.lhaaso_feature_columns + ([file_cfg.lhaaso_time_column] if file_cfg.lhaaso_time_column else [])))

    missing_omni = [col for col in omni_required if col not in df_omni.columns]
    missing_lhaaso = []
    if missing_omni:
        raise KeyError(f"OMNI 数据缺少列: {missing_omni}")
    if missing_lhaaso:
        raise KeyError(f"LHAASO 数据缺少列: {missing_lhaaso}")

    df_omni = df_omni[omni_required + [file_cfg.omni_year_column, file_cfg.omni_dayofyear_column, file_cfg.omni_hour_column]].copy()
    if file_cfg.lhaaso_feature_columns:
        df_lhaaso = _read_table(file_cfg.lhaaso_file_path)
        if file_cfg.lhaaso_time_column not in df_lhaaso.columns:
            raise KeyError(f"LHAASO 数据缺少时间列: {file_cfg.lhaaso_time_column}")
        missing_lhaaso = [col for col in lhaaso_required if col not in df_lhaaso.columns]
        if missing_lhaaso:
            raise KeyError(f"LHAASO 数据缺少列: {missing_lhaaso}")
        df_lhaaso = df_lhaaso[lhaaso_required].copy()

    df_omni["omni_time"] = _build_omni_hourly_time(df_omni, file_cfg)
    df_omni["day_key"] = df_omni["omni_time"].dt.floor("D")

    if file_cfg.lhaaso_feature_columns:
        df_lhaaso["day_key"] = _normalize_daily_time(df_lhaaso[file_cfg.lhaaso_time_column])
        df_merged = pd.merge(df_omni, df_lhaaso.drop(columns=[file_cfg.lhaaso_time_column]), on="day_key", how="inner")
    else:
        df_merged = df_omni.copy()
    df_merged = df_merged.sort_values("omni_time").reset_index(drop=True)

    selected_columns = omni_required + file_cfg.lhaaso_feature_columns
    df_merged = df_merged.dropna(subset=selected_columns).reset_index(drop=True)
    print(f"双尺度对齐后有效小时样本数: {len(df_merged)}")
    return df_merged

In [13]:
# 双时间尺度特征构造

def build_biwindow_features(df_merged: pd.DataFrame, cfg: ExperimentConfig):
    file_cfg = cfg.file_config
    bi_cfg = cfg.biwindow_config
    feat_df = pd.DataFrame(index=df_merged.index)
    feature_names = []

    for item in bi_cfg.omni_hour_lags:
        col = item.var_name
        if col not in df_merged.columns:
            raise KeyError(f"小时级变量不存在: {col}")
        for lag in sorted(set(item.lag_list)):
            feature_name = f"{col}_t-{lag}h"
            feat_df[feature_name] = df_merged[col].shift(lag)
            feature_names.append(feature_name)

    for lag in sorted(set(bi_cfg.target_autoregressive_lags)):
        feature_name = f"target_sw_t-{lag}h"
        feat_df[feature_name] = df_merged[file_cfg.target_column].shift(lag)
        feature_names.append(feature_name)

    if file_cfg.lhaaso_feature_columns and bi_cfg.lhaaso_day_lags:
        daily_lookup = (
            df_merged[["day_key"] + file_cfg.lhaaso_feature_columns]
            .drop_duplicates(subset=["day_key"])
            .sort_values("day_key")
            .set_index("day_key")
        )

        for item in bi_cfg.lhaaso_day_lags:
            col = item.var_name
            if col not in daily_lookup.columns:
                raise KeyError(f"日尺度变量不存在: {col}")
            for lag in sorted(set(item.lag_list)):
                suffix = f"t-{lag}d" if lag > 0 else "t-0d"
                feature_name = f"{col}_{suffix}"
                lookup_index = df_merged["day_key"] - pd.to_timedelta(lag, unit="D")
                feat_df[feature_name] = lookup_index.map(daily_lookup[col])
                feature_names.append(feature_name)

    feat_df["target_sw"] = df_merged[file_cfg.target_column].to_numpy(dtype=float)
    feat_df["time_point"] = df_merged["omni_time"].to_numpy()
    feat_df = feat_df.dropna(axis=0, how="any").reset_index(drop=True)

    X_all = feat_df[feature_names].to_numpy(dtype=float)
    y_all = feat_df["target_sw"].to_numpy(dtype=float)
    time_all = pd.to_datetime(feat_df["time_point"]).to_numpy()
    print(f"双时间尺度滑窗后样本数: {len(X_all)}")
    print(f"最终特征总数: {len(feature_names)}")
    return X_all, y_all, time_all, feature_names

In [14]:
# 数据集切分与标准化

def split_and_scale_dataset(X_all: np.ndarray, y_all: np.ndarray, time_all: np.ndarray, test_rate: float, feature_names: List[str]):
    split_idx = int(len(X_all) * (1 - test_rate))
    if split_idx <= 0 or split_idx >= len(X_all):
        raise ValueError("测试集比例导致训练集或测试集为空")

    Xtr = X_all[:split_idx]
    Xte = X_all[split_idx:]
    ytr = y_all[:split_idx]
    yte = y_all[split_idx:]
    ttr = time_all[:split_idx]
    tte = time_all[split_idx:]

    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(Xtr)
    Xte_scaled = scaler.transform(Xte)

    Xtr_df = pd.DataFrame(Xtr_scaled, columns=feature_names)
    Xte_df = pd.DataFrame(Xte_scaled, columns=feature_names)
    return Xtr_df, Xte_df, ytr, yte, ttr, tte, scaler

In [15]:
# 模型训练与评估

def train_elasticnet_model(Xtr: pd.DataFrame, ytr: np.ndarray, enet_cfg: ElasticNetConfig) -> ElasticNet:
    model = ElasticNet(
        alpha=enet_cfg.alpha,
        l1_ratio=enet_cfg.l1_ratio,
        random_state=enet_cfg.random_state,
        max_iter=10000
    )
    model.fit(Xtr, ytr)
    return model

def evaluate_model(model: ElasticNet, Xtr: pd.DataFrame, Xte: pd.DataFrame, ytr: np.ndarray, yte: np.ndarray):
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)
    baseline_pred = np.full_like(yte, ytr.mean(), dtype=float)
    pearson_r = float(np.corrcoef(yte, yte_pred)[0, 1]) if len(yte) > 1 else float("nan")

    metrics = {
        "train_rmse": float(np.sqrt(mean_squared_error(ytr, ytr_pred))),
        "test_rmse": float(np.sqrt(mean_squared_error(yte, yte_pred))),
        "train_mae": float(mean_absolute_error(ytr, ytr_pred)),
        "test_mae": float(mean_absolute_error(yte, yte_pred)),
        "train_r2": float(r2_score(ytr, ytr_pred)),
        "test_r2": float(r2_score(yte, yte_pred)),
        "test_pearson_r": pearson_r,
        "baseline_test_rmse": float(np.sqrt(mean_squared_error(yte, baseline_pred)))
    }
    return metrics, ytr_pred, yte_pred

def make_coefficient_table(model: ElasticNet, feat_names: List[str]) -> pd.DataFrame:
    coef_df = pd.DataFrame({
        "feature": feat_names,
        "coefficient": model.coef_
    })
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    return coef_df.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

In [16]:
# 图表与残差分析

def plot_feature_coefficients(coef_df: pd.DataFrame, output_dir: Path, top_n: int = 15):
    plot_df = coef_df.head(top_n).sort_values("coefficient")
    plt.figure(figsize=(10, 6), dpi=300)
    colors = ["#c0392b" if v < 0 else "#2980b9" for v in plot_df["coefficient"]]
    plt.barh(plot_df["feature"], plot_df["coefficient"], color=colors)
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.title("ElasticNet Coefficients")
    plt.tight_layout()
    save_path = output_dir / "elasticnet_biwindow_autoreg_coefficients.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    return save_path

def plot_prediction_scatter(y_true: np.ndarray, y_pred: np.ndarray, metrics: dict, output_dir: Path):
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.figure(figsize=(7, 6), dpi=300)
    plt.scatter(y_true, y_pred, alpha=0.7, s=20, c="#2c3e50")
    plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
    text_info = (
        f"R2 = {metrics['test_r2']:.3f}\n"
        f"RMSE = {metrics['test_rmse']:.2f}\n"
        f"r = {metrics['test_pearson_r']:.3f}"
    )
    plt.text(0.03, 0.97, text_info, transform=plt.gca().transAxes, va="top", bbox=dict(boxstyle="round", fc="white", alpha=0.8))
    plt.xlabel("True Solar-Wind Speed (km/s)")
    plt.ylabel("Predicted Solar-Wind Speed (km/s)")
    plt.title("Biwindow ElasticNet Autoreg Scatter")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    save_path = output_dir / "elasticnet_biwindow_autoreg_scatter.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    return save_path

def plot_prediction_series(time_test: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, output_dir: Path):
    plt.figure(figsize=(12, 5), dpi=300)
    plt.plot(time_test, y_true, label="True", lw=1.5, color="#2c3e50")
    plt.plot(time_test, y_pred, label="Predicted", lw=1.5, color="#c0392b")
    plt.xlabel("Time")
    plt.ylabel("Solar-Wind Speed (km/s)")
    plt.title("Biwindow ElasticNet Autoreg Time Series")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    save_path = output_dir / "elasticnet_biwindow_autoreg_timeseries.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    return save_path

def plot_residual_histogram(residual: np.ndarray, output_dir: Path):
    plt.figure(figsize=(7, 5), dpi=300)
    sns.histplot(residual, bins=30, kde=True, color="#34495e")
    plt.xlabel("Residual (True - Predicted)")
    plt.ylabel("Count")
    plt.title("Biwindow Autoreg Residual Histogram")
    plt.tight_layout()
    save_path = output_dir / "elasticnet_biwindow_autoreg_residual_hist.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    return save_path

In [17]:
# SHAP 解释

def run_shap_analysis(model: ElasticNet, Xtr: pd.DataFrame, Xte: pd.DataFrame, output_dir: Path):
    explainer = shap.Explainer(model, Xtr)
    shap_values = explainer(Xte)

    plt.figure(figsize=(10, 8), dpi=300)
    shap.summary_plot(shap_values, Xte, show=False)
    plt.tight_layout()
    summary_path = output_dir / "elasticnet_biwindow_autoreg_shap_summary.png"
    plt.savefig(summary_path, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 8), dpi=300)
    shap.summary_plot(shap_values, Xte, plot_type="bar", show=False)
    plt.tight_layout()
    bar_path = output_dir / "elasticnet_biwindow_autoreg_shap_bar.png"
    plt.savefig(bar_path, bbox_inches="tight")
    plt.close()

    shap_importance = pd.DataFrame({
        "feature": Xte.columns,
        "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_importance_path = output_dir / "elasticnet_biwindow_autoreg_shap_importance.csv"
    shap_importance.to_csv(shap_importance_path, index=False, encoding="utf-8-sig")

    return {
        "summary_path": summary_path,
        "bar_path": bar_path,
        "importance": shap_importance,
        "importance_path": shap_importance_path
    }

In [18]:
# 主流程

def main(config_path: str):
    cfg = load_llm_json_config(config_path)
    output_dir = Path(config_path).resolve().parent.parent / "outputs" / "biwindow_autoreg"
    output_dir.mkdir(parents=True, exist_ok=True)

    df_merged = load_and_merge_biwindow_data(cfg)
    X_all, y_all, time_all, feature_names = build_biwindow_features(df_merged, cfg)
    Xtr, Xte, ytr, yte, time_train, time_test, scaler = split_and_scale_dataset(
        X_all,
        y_all,
        time_all,
        cfg.biwindow_config.test_split_ratio,
        feature_names
    )

    model = train_elasticnet_model(Xtr, ytr, cfg.elasticnet_config)
    metrics, ytr_pred, yte_pred = evaluate_model(model, Xtr, Xte, ytr, yte)
    coef_df = make_coefficient_table(model, feature_names)
    coef_df.to_csv(output_dir / "elasticnet_biwindow_autoreg_coefficients.csv", index=False, encoding="utf-8-sig")
    with open(output_dir / "elasticnet_biwindow_autoreg_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    residual = yte - yte_pred
    figure_paths = {
        "coefficients": plot_feature_coefficients(coef_df, output_dir),
        "scatter": plot_prediction_scatter(yte, yte_pred, metrics, output_dir),
        "timeseries": plot_prediction_series(time_test, yte, yte_pred, output_dir),
        "residual_hist": plot_residual_histogram(residual, output_dir)
    }
    shap_result = run_shap_analysis(model, Xtr, Xte, output_dir)

    print("ElasticNet 双时间尺度自回归训练完成")
    print(f"样本数: {len(X_all)}")
    print(f"训练集 RMSE: {metrics['train_rmse']:.3f}")
    print(f"测试集 RMSE: {metrics['test_rmse']:.3f}")
    print(f"训练集 R2: {metrics['train_r2']:.3f}")
    print(f"测试集 R2: {metrics['test_r2']:.3f}")
    print(f"测试集 Pearson r: {metrics['test_pearson_r']:.3f}")
    print(f"测试集均值基线 RMSE: {metrics['baseline_test_rmse']:.3f}")
    print("系数绝对值 Top 特征:")
    print(coef_df.head(10).to_string(index=False))
    print("SHAP 重要性 Top 特征:")
    print(shap_result['importance'].head(10).to_string(index=False))

    return {
        "config": cfg.model_dump(mode="json"),
        "metrics": metrics,
        "coefficients": coef_df,
        "shap_importance": shap_result['importance'],
        "model": model,
        "scaler": scaler,
        "time_train": time_train,
        "time_test": time_test,
        "y_train": ytr,
        "y_test": yte,
        "y_train_pred": ytr_pred,
        "y_test_pred": yte_pred,
        "figure_paths": figure_paths,
        "shap_paths": {
            "summary": shap_result['summary_path'],
            "bar": shap_result['bar_path'],
            "importance_csv": shap_result['importance_path']
        },
        "output_dir": output_dir
    }

if __name__ == "__main__":
    main(r"D:\astro\ElasticNet\json_Einput\Elasticnet_biwindow_omni_only_autoreg.txt")

双尺度对齐后有效小时样本数: 4369
双时间尺度滑窗后样本数: 4345
最终特征总数: 25


Background dataset has 3041 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=3041 when initializing the masker.


ElasticNet 双时间尺度自回归训练完成
样本数: 4345
训练集 RMSE: 253.937
测试集 RMSE: 380.291
训练集 R2: 0.024
测试集 R2: -0.001
测试集 Pearson r: 0.039
测试集均值基线 RMSE: 380.042
系数绝对值 Top 特征:
           feature  coefficient  abs_coefficient
    target_sw_t-1h     9.740580         9.740580
    target_sw_t-2h     9.735661         9.735661
    target_sw_t-3h     9.583300         9.583300
    target_sw_t-6h     9.313979         9.313979
   target_sw_t-12h     8.343280         8.343280
BY, nT (GSE)_t-12h    -8.115327         8.115327
   target_sw_t-24h     6.096950         6.096950
 Scalar B, nT_t-6h     5.051193         5.051193
 BZ, nT (GSE)_t-4h    -3.146723         3.146723
 Scalar B, nT_t-5h     2.805430         2.805430
SHAP 重要性 Top 特征:
           feature  mean_abs_shap
BY, nT (GSE)_t-12h       6.615129
 Scalar B, nT_t-6h       4.048021
   target_sw_t-24h       3.242962
    target_sw_t-1h       2.456680
    target_sw_t-2h       2.437810
 BZ, nT (GSE)_t-4h       2.430496
    target_sw_t-3h       2.412343
    target_sw_t-